<a href="https://colab.research.google.com/github/Truptinaikwadi/Gen-Ai-task-assesment/blob/main/Task_5_Fine_tune_a_transformer_model_(DistilBERT)_to_perform__Part_of_Speech_(POS)_Tagging_Chunking_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assignment: Fine-Tuning BERT for POS Tagging & Chunking**



**Objective**

Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.




STEP 1: Install Libraries

In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate


STEP 2: Import Libraries

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed
)
from pprint import pprint


STEP 3: Setup Device & Seed

In [ ]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


STEP 4: Load Dataset (POS Tagging)

In [32]:
from datasets import load_dataset

for name in [
    "Babelscape/conll2003-nemo",
    "conll2003",
    "wikiann",
]:
    try:
        if name == "wikiann":
            dataset = load_dataset("wikiann", "en")
        else:
            dataset = load_dataset(name)

        print(f"✅ Loaded dataset: {name}")
        break

    except Exception as e:
        print(f"⚠️ Could not load {name}: {e}")

else:
    raise RuntimeError("❌ Unable to load any dataset")

✅ Loaded dataset: wikiann


STEP 6: Load Tokenizer

In [33]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


STEP 5: Identify Labels

In [34]:
label_col = "ner_tags"

label_list = dataset["train"].features[label_col].feature.names
num_labels = len(label_list)

print("Labels:", label_list)

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [35]:
small_train = dataset["train"].select(range(2000))
small_val = dataset["validation"].select(range(500))

STEP 7: Tokenization + Label Alignment

In [ ]:
def align_labels_with_tokens(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        word_labels = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                word_labels.append(-100)
            elif word_idx != previous_word_idx:
                word_labels.append(examples[label_col][i][word_idx])
            else:
                word_labels.append(-100)

            previous_word_idx = word_idx

        labels.append(word_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

small_train = small_train.map(align_labels_with_tokens, batched=True)
small_val = small_val.map(align_labels_with_tokens, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
sample = small_train[0]

print("Input IDs:", sample["input_ids"])
print("Attention Mask:", sample["attention_mask"])
print("Labels:", sample["labels"])

STEP 8: Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

STEP 9: Load Model

In [ ]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

STEP 10: Evaluation Metrics


In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

STEP 11: Training Arguments

In [ ]:
args = TrainingArguments(
    output_dir="ner_model",
    eval_strategy="epoch",   # your version
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,      # FAST
    max_steps=500,           # LIMIT
    weight_decay=0.01,
)

STEP 12: Trainer Setup

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

STEP 13: Train Model

In [ ]:
trainer.train()

STEP 14: Evaluate Model

In [ ]:
trainer.evaluate()


STEP 15: Inference

In [ ]:
nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

text = "Ram got Data Analyst job ,John works at Google in California"
output = nlp(text)

for token in output:
    print(token)